In [0]:
%sh
pwd
cd ..
pwd
cd ../..
pwd

In [0]:
%sh
git --version
cd ..
ls -la

cd ../..
ls -la

In [0]:
%sh
cd ..

find . -maxdepth 5 -type f

In [0]:
# clean memory
dbutils.library.restartPython()

In [0]:
# Cell 1: 开启自动重载神器
%load_ext autoreload
%autoreload 2

import sys
import os
# 把项目的 src 目录临时加入环境变量，这样就能直接 from bronze... 导入了
# 2. 动态注入环境变量（就等于你在集群里配置了）
os.environ["ENV_CATALOG"] = "nyc"  # 请换成你真实的 catalog 名字
os.environ["ENV_SILVER_DB"] = "process_bronze" # 可选，如果你想临时换个 schema 测试

# 3. 把 src 加入路径
sys.path.append(os.path.abspath("../src")) 

print(f"Current Catalog has been setted to: {os.getenv('ENV_CATALOG')}")

In [0]:
# Cell 2: 手工运行与调试
from pyspark.sql import SparkSession
from bronze.loader import TaxiBronzeLoader

# 获取当前环境的 Spark
spark = SparkSession.builder.getOrCreate()

# 实例化你的 Loader
loader = TaxiBronzeLoader(spark)

# 直接调用里面的小方法来测试，比如只测推演路径的方法
#paths = loader._generate_target_paths(201001, 201002)
#print(paths)

# 或者跑全流程
loader.write_idempotent(start_time=201001, end_time=201002)

In [0]:
%sql
select * from nyc.process_silver.pipeline_metrics 

In [0]:
import sys
import os
import pytest

# 2. 封印字节码缓存 (解决之前的 Errno 95 权限报错)
sys.dont_write_bytecode = True

# 3. 智能路径导航：确保我们站在项目根目录下执行
current_dir = os.getcwd()
if current_dir.endswith("notebooks"):
    os.chdir("..")
    print(f"✅ 已切换到根目录: {os.getcwd()}")

# 4. 把根目录注入到 Python 视野中 (替代 PYTHONPATH=.)
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

# 5. 在拥有完整高级权限的 Python 进程中，直接引爆 Pytest！
print("🚀 启动工业级单元测试流水线 (单测精准狙击模式)...\n" + "="*40)

retcode = pytest.main([
    # 核心修改：用具体的路径和节点，替换掉原来的 "tests/"
    #"tests/test_bronze_transformations.py::TestSilverIntegration::test_full_pipeline_and_partition_overwrite", 
    "tests/test_bronze_transformations.py", 
    "-vv",               # 极度详细模式
    "--tb=long",         # 显示完整的错误堆栈，不要缩略
    "-s",                # 允许打印 print 语句
    "-p", "no:cacheprovider"
])

In [0]:
import os
import sys
import pytest

# ==================================================
# 1. 禁止生成 __pycache__
# ==================================================
sys.dont_write_bytecode = True

# ==================================================
# 2. 切换到项目根目录
# ==================================================
current_dir = os.getcwd()
if current_dir.endswith("notebooks"):
    os.chdir("..")

project_root = os.getcwd()
print(f"📁 当前项目根目录: {project_root}")

# ==================================================
# 3. 注入 PYTHONPATH (🌟 核心修复 1：必须指向 src！)
# ==================================================
src_dir = os.path.join(project_root, "src")

if src_dir not in sys.path:
    sys.path.insert(0, src_dir) # 强行让 Python 从 src 内部开始寻找模块

print("\n📦 Python Path:\n")
for p in sys.path[:5]:
    print(p)

# ==================================================
# 4. 验证 bronze package (🌟 核心修复 2：加上 src 物理路径)
# ==================================================
print("\n🔍 检查 bronze package...\n")

# 在系统文件寻找层面，我们必须带上 src
bronze_path = os.path.join(project_root, "src", "bronze")
if os.path.exists(bronze_path):
    print(os.listdir(bronze_path))
else:
    print(f"❌ 警告：找不到物理路径 {bronze_path}，请检查目录结构！")

# ==================================================
# 5. 启动 pytest
# ==================================================
print(
    "\n🚀 启动工业级 Spark Unit Test Pipeline\n"
    + "=" * 60
)
retcode = pytest.main([
    "tests/test_bronze_transformations.py",
    "-vv",
    "--tb=long",
    "-s",
    "-p", "no:cacheprovider"
])

# ==================================================
# 6. 输出结果
# ==================================================
print("\n" + "=" * 60)
if retcode == 0:
    print("✅ 所有测试通过")
else:
    print(f"❌ 测试失败，退出码: {retcode}")

In [0]:
import os
import sys
import pytest

# ==================================================
# 1. 禁止生成 __pycache__
# ==================================================
sys.dont_write_bytecode = True

# ==================================================
# 2. 切换到项目根目录
# ==================================================
current_dir = os.getcwd()
if current_dir.endswith("notebooks"):
    os.chdir("..")

project_root = os.getcwd()
print(f"📁 当前项目根目录: {project_root}")

# ==================================================
# 3. 注入 PYTHONPATH (🌟 核心修复 1：必须指向 src！)
# ==================================================
src_dir = os.path.join(project_root, "src")

if src_dir not in sys.path:
    sys.path.insert(0, src_dir) # 强行让 Python 从 src 内部开始寻找模块

print("\n📦 Python Path:\n")
for p in sys.path[:5]:
    print(p)

# ==================================================
# 4. 验证 bronze package (🌟 核心修复 2：加上 src 物理路径)
# ==================================================
print("\n🔍 检查 bronze package...\n")

# 在系统文件寻找层面，我们必须带上 src
bronze_path = os.path.join(project_root, "src", "bronze")
if os.path.exists(bronze_path):
    print(os.listdir(bronze_path))
else:
    print(f"❌ 警告：找不到物理路径 {bronze_path}，请检查目录结构！")

# ==================================================
# 5. 启动 pytest
# ==================================================
print(
    "\n🚀 启动工业级 Spark Unit Test Pipeline\n"
    + "=" * 60
)
retcode = pytest.main([
    "tests/test_bronze_unit.py",
    "-vv",
    "--tb=long",
    "-s",
    "-p", "no:cacheprovider"
])

# ==================================================
# 6. 输出结果
# ==================================================
print("\n" + "=" * 60)
if retcode == 0:
    print("✅ 所有测试通过")
else:
    print(f"❌ 测试失败，退出码: {retcode}")